# RL Trading Bot - XAU/USD 1H - Entrenamiento en Colab

Notebook para entrenar el modelo PPO en Google Colab.

**Pasos:**
1. Clonar repo desde GitHub
2. Instalar dependencias
3. Benchmark de velocidad (CPU vs GPU)
4. Entrenar modelo (2M steps)
5. Evaluar en test set
6. Guardar resultados en Google Drive
7. Diagnostico de trades
8. Visualizacion de resultados

## 1. Clonar repo desde GitHub

In [ ]:
import os

REPO_URL = "https://github.com/Valen89hh/bot-perdedor.git"
LOCAL_PROJECT_PATH = "/content/rl-trading-bot"

# Clonar (o pull si ya existe)
if os.path.exists(LOCAL_PROJECT_PATH):
    !cd {LOCAL_PROJECT_PATH} && git pull
else:
    !git clone {REPO_URL} {LOCAL_PROJECT_PATH}

os.chdir(LOCAL_PROJECT_PATH)
print(f"Proyecto en {LOCAL_PROJECT_PATH}")
print(f"Archivos: {os.listdir('.')}")

## 2. Instalar dependencias

In [ ]:
!pip install -q stable-baselines3[extra] gymnasium scikit-learn tensorboard
!pip install -q --no-deps pandas-ta

## 3. Verificar GPU y benchmark de velocidad

Compara steps/segundo entre CPU y GPU para elegir el device optimo.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
import time
import sys
sys.path.insert(0, LOCAL_PROJECT_PATH)

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from src.utils.data_loader import load_data, split_data
from src.environment.features import compute_features, normalize_features
from src.environment.trading_env import TradingEnv
from config import CONFIG

# Preparar datos una sola vez
df = load_data(CONFIG["data_path"])
train_df, val_df, _ = split_data(df, CONFIG["train_ratio"], CONFIG["val_ratio"])
feat = compute_features(train_df)
feat_norm, _ = normalize_features(feat)
prices = train_df.loc[feat_norm.index]

def make_env():
    def _init():
        env = TradingEnv(
            df_features=feat_norm, df_prices=prices,
            initial_balance=10000, commission=0.0002, max_position_size=1,
        )
        return Monitor(env)
    return _init

BENCH_STEPS = 20_000
results = {}

for device in ["cpu", "cuda"]:
    for n_envs in [1, 4]:
        label = f"{device.upper()} DummyVecEnv x{n_envs}"
        env = DummyVecEnv([make_env() for _ in range(n_envs)])
        model = PPO(
            "MlpPolicy", env, device=device, verbose=0,
            n_steps=2048, batch_size=64,
            policy_kwargs={"net_arch": [128, 64]},
        )
        start = time.time()
        model.learn(total_timesteps=BENCH_STEPS)
        elapsed = time.time() - start
        sps = BENCH_STEPS / elapsed
        results[label] = sps
        print(f"{label:30s} -> {sps:>6.0f} steps/sec  ({elapsed:.1f}s)")

best = max(results, key=results.get)
print(f"\n>>> Mas rapido: {best} ({results[best]:.0f} steps/sec)")
print(f">>> Tiempo estimado 2M steps: {2_000_000 / results[best] / 60:.0f} minutos")

## 4. Entrenar modelo (2M steps)

Entrena desde cero con Discrete(3): HOLD / OPEN LONG / CLOSE LONG.

In [ ]:
from src.agent.train import train

start = time.time()
model = train()
elapsed = time.time() - start

total_steps = CONFIG["total_timesteps"]
print(f"\nEntrenamiento completado en {elapsed/60:.1f} minutos")
print(f"Velocidad promedio: {total_steps/elapsed:.0f} steps/sec")

## 5. Evaluar y comparar modelos (best vs final)

Evalua `best_model.zip` y `final_model.zip` en train y test, y muestra los resultados lado a lado.

In [ ]:
import numpy as np
from src.agent.evaluate import evaluate

models_dir = CONFIG["models_dir"]
logs_dir = CONFIG["logs_dir"]

# --- Info de steps y eval reward desde evaluations.npz ---
eval_npz_path = os.path.join(logs_dir, "evaluations.npz")
best_step, best_eval_reward = "N/A", "N/A"
final_step = f"{CONFIG['total_timesteps']:,}"

if os.path.exists(eval_npz_path):
    eval_data = np.load(eval_npz_path)
    timesteps = eval_data["timesteps"]
    mean_rewards = eval_data["results"].mean(axis=1)
    best_idx = np.argmax(mean_rewards)
    best_step = f"{int(timesteps[best_idx]):,}"
    best_eval_reward = f"{mean_rewards[best_idx]:.2f}"
    final_eval_reward = f"{mean_rewards[-1]:.2f}"
    print(f"best_model.zip  -> guardado en step {best_step}, eval reward: {best_eval_reward}")
    print(f"final_model.zip -> guardado en step {final_step}, eval reward: {final_eval_reward}")
else:
    final_eval_reward = "N/A"
    print("evaluations.npz no encontrado, no se puede determinar step/reward info")

# --- Evaluar ambos modelos en test y train ---
best_path = os.path.join(models_dir, "best_model.zip")
final_path = os.path.join(models_dir, "final_model.zip")

results = {}
for name, path in [("best_model", best_path), ("final_model", final_path)]:
    if not os.path.exists(path):
        print(f"\n{name} no encontrado en {path}, saltando...")
        continue
    for ds in ["test", "train"]:
        print(f"\n{'='*60}")
        print(f"  Evaluando {name} en {ds.upper()}")
        print(f"{'='*60}")
        r = evaluate(model_path=path, dataset=ds)
        results[(name, ds)] = r["metrics"]
        results[(name, ds)]["buy_hold"] = r["buy_hold_return_pct"]

# --- Tabla comparativa ---
print("\n\n" + "=" * 80)
print("  COMPARACION: best_model vs final_model")
print("=" * 80)

header = f"{'Metrica':<20} {'best (test)':>14} {'final (test)':>14} {'best (train)':>14} {'final (train)':>14}"
print(header)
print("-" * 80)

metrics_keys = [
    ("Return %",        "total_return_pct",  ".2f"),
    ("Buy & Hold %",    "buy_hold",          ".2f"),
    ("Sharpe Ratio",    "sharpe_ratio",      ".4f"),
    ("Max Drawdown %",  "max_drawdown_pct",  ".2f"),
    ("Win Rate %",      "win_rate_pct",      ".2f"),
    ("Profit Factor",   "profit_factor",     ".4f"),
    ("Total Trades",    "total_trades",      "d"),
    ("Final Balance",   "final_balance",     ",.2f"),
]

for label, key, fmt in metrics_keys:
    vals = []
    for model_name in ["best_model", "final_model"]:
        for ds in ["test", "train"]:
            if (model_name, ds) in results:
                v = results[(model_name, ds)][key]
                vals.append(f"{v:{fmt}}")
            else:
                vals.append("N/A")
    prefix = "$" if key == "final_balance" else ""
    print(f"  {label:<18} {prefix}{vals[0]:>13} {prefix}{vals[1]:>13} {prefix}{vals[2]:>13} {prefix}{vals[3]:>13}")

print("-" * 80)
print(f"  {'Step guardado':<18} {best_step:>14} {final_step:>14} {'-':>14} {'-':>14}")
print(f"  {'Eval Reward':<18} {best_eval_reward:>14} {final_eval_reward:>14} {'-':>14} {'-':>14}")
print("=" * 80)

## 6. Guardar resultados en Google Drive

Monta Drive y copia modelos, logs y resultados.

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/TECSUP/rl-trading-bot"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

folders_to_sync = ["models", "results", "logs"]

for folder in folders_to_sync:
    src = os.path.join(LOCAL_PROJECT_PATH, folder)
    dst = os.path.join(DRIVE_SAVE_PATH, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Sincronizado: {folder} -> Drive")

print("\nTodo guardado en Google Drive.")

## 7. Diagnostico rapido de trades

In [ ]:
import json
import numpy as np

for ds in ["test", "train"]:
    path = os.path.join(LOCAL_PROJECT_PATH, "results", f"evaluation_{ds}.json")
    if not os.path.exists(path):
        print(f"No se encontro {path}, saltando...")
        continue
    with open(path) as f:
        data = json.load(f)

    trades = data["trade_history"]
    m = data["metrics"]
    opens = [t for t in trades if t["type"] == "open_long"]
    closes = [t for t in trades if t["type"] == "close_long"]
    pnls = [t["pnl"] for t in closes]
    winners = [p for p in pnls if p > 0]
    losers = [p for p in pnls if p < 0]

    print(f"\n{'='*50}")
    print(f"  {ds.upper()} SET")
    print(f"{'='*50}")
    print(f"  Return:        {m['total_return_pct']:>8.2f}%")
    print(f"  Buy & Hold:    {data['buy_hold_return_pct']:>8.2f}%")
    print(f"  Sharpe:        {m['sharpe_ratio']:>8.4f}")
    print(f"  Max Drawdown:  {m['max_drawdown_pct']:>8.2f}%")
    print(f"  Win Rate:      {m['win_rate_pct']:>8.2f}%")
    print(f"  Total Trades:  {m['total_trades']:>8d}")
    print(f"  Profit Factor: {m['profit_factor']:>8.4f}")
    print(f"  ---")
    print(f"  Long opens:    {len(opens)}")
    if winners:
        print(f"  Avg winner:    ${np.mean(winners):.2f}")
    if losers:
        print(f"  Avg loser:     ${np.mean(losers):.2f}")
        print(f"  Max loser:     ${min(losers):.2f}")
    if winners and losers:
        print(f"  W/L ratio:     {abs(np.mean(winners)/np.mean(losers)):.2f}")

## 8. Visualizacion de resultados

In [ ]:
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Cargar resultados
with open(os.path.join(LOCAL_PROJECT_PATH, "results", "evaluation_test.json")) as f:
    data = json.load(f)

prices = data["prices"]
trades = data["trade_history"]
metrics = data["metrics"]

# --- Grafico de velas + trades + portfolio ---
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.7, 0.3],
    subplot_titles=["XAU/USD + Trades", "Portfolio Value"],
    vertical_spacing=0.08
)

# Velas japonesas
fig.add_trace(go.Candlestick(
    x=prices["datetime"],
    open=prices["open"], high=prices["high"],
    low=prices["low"], close=prices["close"],
    name="XAU/USD"
), row=1, col=1)

# Marcar trades (BUY = verde, SELL = rojo)
buys = [t for t in trades if t["type"] == "open_long"]
sells = [t for t in trades if t["type"] == "close_long"]

if buys:
    fig.add_trace(go.Scatter(
        x=[t["timestamp"] for t in buys],
        y=[t["price"] for t in buys],
        mode="markers", name="BUY",
        marker=dict(symbol="triangle-up", size=10, color="green")
    ), row=1, col=1)

if sells:
    fig.add_trace(go.Scatter(
        x=[t["timestamp"] for t in sells],
        y=[t["price"] for t in sells],
        mode="markers", name="SELL",
        marker=dict(symbol="triangle-down", size=10, color="red")
    ), row=1, col=1)

# Portfolio
fig.add_trace(go.Scatter(
    x=data["timestamps"][:len(data["portfolio_history"])],
    y=data["portfolio_history"],
    name="Portfolio", line=dict(color="blue")
), row=2, col=1)

fig.update_layout(
    height=800, title_text=f"RL Trading Bot | Return: {metrics['total_return_pct']:.2f}% | "
    f"Sharpe: {metrics['sharpe_ratio']:.4f} | Win Rate: {metrics['win_rate_pct']:.1f}% | "
    f"Trades: {metrics['total_trades']}",
    xaxis_rangeslider_visible=False, showlegend=True
)
fig.show()